In [2]:
from pyspark.sql import SparkSession
import os

# Nạp đầy đủ Package cho Kafka, Iceberg, Nessie và S3 (MinIO)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1,'
    'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,'
    'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.4_2.12:0.67.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4 '  # THƯ VIỆN CÒN THIẾU Ở ĐÂY
    'pyspark-shell'
)

spark = SparkSession.builder \
    .appName("Fix-S3A-Error") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "main") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://silver/") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

# 2. Đọc bảng bằng Spark SQL
df = spark.sql("SELECT * FROM nessie.silver_db.amazon_purchase_silver")

# 3. Chuyển đổi sang Pandas và hiển thị (đẹp hơn show())
# Lấy 10 dòng đầu tiên để xem nhanh
pdf = df.limit(5).toPandas()

In [3]:
pdf

,order_date,unit_price,quantity,state,product_title,product_code,category,survey_id,processed_at
0,2019-05-08,27.59,1.0,FL,5 Pack Identify Diagnostics 12 Panel Drug Test...,B01DMMPO58,HEALTH_PERSONAL_CARE,R_1DLRE54vbCjvb6t,2026-04-14 17:15:28.286
1,2019-05-08,67.15,1.0,NC,"Swingline GBC Binding System, Manual, Desktop ...",B00006IAS3,OFFICE_PRODUCTS,R_1OK73VTU4KtCpkI,2026-04-14 17:15:28.286
2,2019-05-08,66.30,1.0,OH,Amazon Basics Double Hammock with 9-Foot Space...,B01LQV4ML4,HAMMOCK,R_2XdMB4tf1qMJDlU,2026-04-14 17:15:28.286
3,2019-05-08,19.99,1.0,IN,"Salt, Fat, Acid, Heat: Mastering the Elements ...",1476753830,ABIS_BOOK,R_31QS1Ms6iQEU3mI,2026-04-14 17:15:28.286
4,2019-05-08,16.49,1.0,NY,Dr. Bronner's - Organic Lip Balm (Peppermint.1...,B001ET70SG,LIP_BALM,R_e8KTElZZLWryLWp,2026-04-14 17:15:28.286


In [4]:
from pyspark.sql.functions import count

# Tổng số dòng
total_count = df.count()

# Số dòng distinct
distinct_count = df.distinct().count()

print("Total rows:", total_count)
print("Distinct rows:", distinct_count)
print("Duplicate rows:", total_count - distinct_count)

Total rows: 1850717
Distinct rows: 1839093
Duplicate rows: 11624


In [5]:
from pyspark.sql.functions import col

duplicates = (
    df.groupBy(df.columns)
      .count()
      .filter(col("count") > 1)
      .orderBy(col("count").desc())
)

# Chỉ lấy 50 dòng đầu rồi mới chuyển
pdf = duplicates.limit(10).toPandas()

pdf

,order_date,unit_price,quantity,state,product_title,product_code,category,survey_id,processed_at,count
0,2023-03-10,1.01,1.0,"""NaN""",Amazon.com eGift Card,B06XWXTWXT,GIFT_CARD,R_tPBONZU30Yotued,2026-04-14 17:15:28.286,59
1,2021-08-04,5.00,1.0,"""NaN""",Amazon.com Gift Card Balance Reload,B00IX1I3G6,GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,56
2,2021-11-30,1.01,1.0,"""NaN""",Amazon Reload,B086KKT3RX,GIFT_CARD,R_24DbIDhFrNpJkdz,2026-04-14 17:15:28.286,50
3,2021-08-03,5.00,1.0,"""NaN""",Amazon.com Gift Card Balance Reload,B00IX1I3G6,GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,43
4,2021-09-01,5.00,1.0,"""NaN""",Amazon Reload,B086KKT3RX,ABIS_GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,39
5,2020-06-09,1.00,1.0,"""NaN""",Group Gift Wedding Contribution,B071DPM9RD,GIFT_CARD,R_31ADYMez64zKe2X,2026-04-14 17:15:28.286,32
6,2023-01-31,1.01,1.0,"""NaN""",Amazon.com eGift Card,B09F1RMT8W,ELECTRONIC_GIFT_CARD,R_tPBONZU30Yotued,2026-04-14 17:15:28.286,31
7,2021-08-29,5.00,1.0,"""NaN""",Amazon Reload,B086KKT3RX,ABIS_GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,30
8,2021-08-10,5.00,1.0,"""NaN""",Amazon.com Gift Card Balance Reload,B00IX1I3G6,GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,29
9,2021-08-17,5.00,1.0,"""NaN""",Amazon Reload,B086KKT3RX,ABIS_GIFT_CARD,R_SDxHC9YZwYTKedz,2026-04-14 17:15:28.286,29
